## The State of Tax Justice: Estimate misalignment for 2021

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
# Reset display.max_rows to default
pd.reset_option('display.max_rows')


pd.options.display.float_format = '{:,.8f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2021

26
38
46
50
52
52


,iso_parent,year
0,ARE,2021
155,ARG,2021
269,AUS,2021
765,AUT,2021
797,AZE,2021
839,BEL,2021
1008,BGR,2021
1028,BHR,2021
1068,BMU,2021
1649,BRA,2021


### Step 1.2. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [8]:
misalignment_2021 = cbcr_sample[cbcr_sample['year'] == 2021].copy()

# Calculate the misalignment for 2021
misalignment_2021 = calculate_misalignment(misalignment_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2021.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2021 = misalignment_2021[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2021[misalignment_2021['iso_partner'] == 'USA']

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_12769/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
140,ARE,USA,2021,"-1,211,834,974.55800104","1,676,278,193.97889853","-566,015,448.00000000","28,384.00000000","22,510,101,353.00000000","11,019,650,222.00000000","1,566,840,057.21600008","33,844,332,143.00000000","25,838,686,878.00000000","3,328,585,524.00000000",19.00000000
165,ARG,USA,2021,0.00000000,"9,480,880.09163740","67,155,125.00000000",114.00000000,"113,406,031.00000000","103,940,740.00000000","6,292,973.73600000","1,762,118,722.00000000","132,596,801.00000000","19,190,770.00000000",2.00000000
245,AUS,USA,2021,0.00000000,"7,200,422,731.80306339","8,095,396,499.00000000","89,401.00000000","57,209,834,217.00000000","37,039,556,556.00000000","4,935,071,447.12400055","174,000,000,000.00000000","73,738,510,494.00000000","16,528,676,277.00000000",27.00000000
285,AZE,USA,2021,"-8,636,245.39224626","10,751,113.69069204","-5,934,134.00000000",21.00000000,"748,376,581.00000000","7,712,734.00000000","1,159,232.00400000","20,001,000.00000000","751,937,979.00000000","3,561,398.00000000",NaN
314,BEL,USA,2021,"-553,930,903.10677862","3,029,537,331.16806984","2,376,700,000.00000000","58,400.00000000","43,812,500,000.00000000","14,989,500,000.00000000","3,223,769,001.60000038","380,000,000,000.00000000","54,828,300,000.00000000","11,015,800,000.00000000",25.00000000
353,BHR,USA,2021,0.00000000,"5,748,649.02508260","28,628,903.00000000",46.00000000,"31,352,002.00000000","494,407.00000000","2,539,270.10400000","1,253,597.00000000","491,165,142.00000000","459,813,140.00000000",NaN
441,BMU,USA,2021,"-2,631,136,978.25879192","11,377,613,393.12938690","8,368,805,189.00000000","95,211.00000000","87,810,248,119.00000000","43,717,634,640.00000000","5,255,792,301.56400108","75,203,353,654.00000000","109,000,000,000.00000000","21,625,977,894.00000000",16.00000000
480,BRA,USA,2021,"-1,561,921,714.71748805","12,227,561,481.76749611","10,586,870,772.00000000","121,286.00000000","68,461,937,528.00000000","17,087,060,697.00000000","6,695,172,039.86400032","47,722,813,871.00000000","79,304,813,607.00000000","10,842,876,078.00000000",20.00000000
493,CAN,USA,2021,"-62,835,521,174.61859131","120,748,616,174.61856079","57,913,095,000.00000000","867,200.00000000","448,000,000,000.00000000","440,000,000,000.00000000","47,870,761,612.80000305","1,150,000,000,000.00000000","561,000,000,000.00000000","112,000,000,000.00000000",NaN
623,CHE,USA,2021,"-2,611,278,125.47509575","17,075,485,567.54924774","13,563,295,251.00000000","303,765.00000000","267,000,000,000.00000000","91,543,939,101.00000000","16,768,290,937.86000443","398,000,000,000.00000000","334,000,000,000.00000000","67,308,695,757.00000000",64.00000000


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [9]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2021['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2021['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2021['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2021['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2021['payroll'].sum()
total_stated_capital = misalignment_2021['stated_capital'].sum()
total_total_revenues = misalignment_2021['total_revenues'].sum()
total_related_party_revenues = misalignment_2021['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2021['holding_or_managing_ip'].sum()

misalignment_2021['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2021['total_n_employees'] = total_n_employees
misalignment_2021['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2021['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2021['total_payroll'] = total_payroll
misalignment_2021['total_stated_capital'] = total_stated_capital
misalignment_2021['total_total_revenues'] = total_total_revenues
misalignment_2021['total_related_party_revenues'] = total_related_party_revenues
misalignment_2021['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2021.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2021.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2021.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2021.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2021.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2021.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2021.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2021.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2021.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2021 dataframe
misalignment_2021 = misalignment_2021.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2021[misalignment_2021['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
140,ARE,USA,2021,"-1,211,834,974.55800104","1,676,278,193.97889853","-566,015,448.00000000","28,384.00000000","22,510,101,353.00000000","11,019,650,222.00000000","1,566,840,057.21600008","33,844,332,143.00000000","25,838,686,878.00000000","3,328,585,524.00000000",19.00000000,"7,762,091,416,141.00000000","157,545,260.00000000","61,215,345,239,302.00000000","42,955,732,312,253.00000000","4,519,767,206,421.00683594","107,533,344,406,981.00000000","86,567,376,841,818.00000000","25,321,553,863,775.00000000","22,468.00000000","2,091,232,518,307.00000000","30,088,947.00000000","16,720,675,917,579.00000000","9,446,740,674,823.00000000","1,660,955,729,955.22802734","22,120,338,504,730.00000000","22,139,101,840,847.00000000","5,389,605,935,829.00000000","1,503.00000000"
165,ARG,USA,2021,0.00000000,"9,480,880.09163740","67,155,125.00000000",114.00000000,"113,406,031.00000000","103,940,740.00000000","6,292,973.73600000","1,762,118,722.00000000","132,596,801.00000000","19,190,770.00000000",2.00000000,"7,762,091,416,141.00000000","157,545,260.00000000","61,215,345,239,302.00000000","42,955,732,312,253.00000000","4,519,767,206,421.00683594","107,533,344,406,981.00000000","86,567,376,841,818.00000000","25,321,553,863,775.00000000","22,468.00000000","2,091,232,518,307.00000000","30,088,947.00000000","16,720,675,917,579.00000000","9,446,740,674,823.00000000","1,660,955,729,955.22802734","22,120,338,504,730.00000000","22,139,101,840,847.00000000","5,389,605,935,829.00000000","1,503.00000000"
245,AUS,USA,2021,0.00000000,"7,200,422,731.80306339","8,095,396,499.00000000","89,401.00000000","57,209,834,217.00000000","37,039,556,556.00000000","4,935,071,447.12400055","174,000,000,000.00000000","73,738,510,494.00000000","16,528,676,277.00000000",27.00000000,"7,762,091,416,141.00000000","157,545,260.00000000","61,215,345,239,302.00000000","42,955,732,312,253.00000000","4,519,767,206,421.00683594","107,533,344,406,981.00000000","86,567,376,841,818.00000000","25,321,553,863,775.00000000","22,468.00000000","2,091,232,518,307.00000000","30,088,947.00000000","16,720,675,917,579.00000000","9,446,740,674,823.00000000","1,660,955,729,955.22802734","22,120,338,504,730.00000000","22,139,101,840,847.00000000","5,389,605,935,829.00000000","1,503.00000000"
285,AZE,USA,2021,"-8,636,245.39224626","10,751,113.69069204","-5,934,134.00000000",21.00000000,"748,376,581.00000000","7,712,734.00000000","1,159,232.00400000","20,001,000.00000000","751,937,979.00000000","3,561,398.00000000",NaN,"7,762,091,416,141.00000000","157,545,260.00000000","61,215,345,239,302.00000000","42,955,732,312,253.00000000","4,519,767,206,421.00683594","107,533,344,406,981.00000000","86,567,376,841,818.00000000","25,321,553,863,775.00000000","22,468.00000000","2,091,232,518,307.00000000","30,088,947.00000000","16,720,675,917,579.00000000","9,446,740,674,823.00000000","1,660,955,729,955.22802734","22,120,338,504,730.00000000","22,139,101,840,847.00000000","5,389,605,935,829.00000000","1,503.00000000"
314,BEL,USA,2021,"-553,930,903.10677862","3,029,537,331.16806984","2,376,700,000.00000000","58,400.00000000","43,812,500,000.00000000","14,989,500,000.00000000","3,223,769,001.60000038","380,000,000,000.000

### Step 4.2. Calculate the shares for all the variables

In [10]:
# Final Misalignment
final_misalignment_2021 = misalignment_2021

# Calculate the shares for all variables
final_misalignment_2021['share_reported_total_profit_loss_by_partner'] = misalignment_2021['total_profit_loss_by_partner'] / misalignment_2021['total_profit_loss_before_income_tax_corrected']
final_misalignment_2021['share_reported_total_n_employees_by_partner'] = misalignment_2021['total_n_employees_by_partner'] / misalignment_2021['total_n_employees']
final_misalignment_2021['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2021['total_unrelated_party_revenues_by_partner'] / misalignment_2021['total_unrelated_party_revenues']
final_misalignment_2021['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2021['total_tangible_assets_except_cash_by_partner'] / misalignment_2021['total_tangible_assets_except_cash']
final_misalignment_2021['share_reported_total_payroll_by_partner'] = misalignment_2021['total_payroll_by_partner'] / misalignment_2021['total_payroll']
final_misalignment_2021['share_reported_total_stated_capital_by_partner'] = misalignment_2021['total_stated_capital_by_partner'] / misalignment_2021['total_stated_capital']
final_misalignment_2021['share_reported_total_total_revenues_by_partner'] = misalignment_2021['total_total_revenues_by_partner'] / misalignment_2021['total_total_revenues']
final_misalignment_2021['share_reported_total_related_party_revenues_by_partner'] = misalignment_2021['total_related_party_revenues_by_partner'] / misalignment_2021['total_related_party_revenues']
final_misalignment_2021['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2021['total_holding_or_managing_ip_by_partner'] / misalignment_2021['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2021[final_misalignment_2021['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
140,ARE,USA,2021,"-1,211,834,974.56","1,676,278,193.98","-566,015,448.00","28,384.00","22,510,101,353.00","11,019,650,222.00","1,566,840,057.22","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
165,ARG,USA,2021,0.00,"9,480,880.09","67,155,125.00",114.00,"113,406,031.00","103,940,740.00","6,292,973.74","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
245,AUS,USA,2021,0.00,"7,200,422,731.80","8,095,396,499.00","89,401.00","57,209,834,217.00","37,039,556,556.00","4,935,071,447.12","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
285,AZE,USA,2021,"-8,636,245.39","10,751,113.69","-5,934,134.00",21.00,"748,376,581.00","7,712,734.00","1,159,232.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
314,BEL,USA,2021,"-553,930,903.11","3,029,537,331.17","2,376,700,000.00","58,400.00","43,812,500,000.00","14,989,500,000.00","3,223,769,001.60","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)

In [11]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2021 = final_misalignment_2021[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2021 = shares_reported_2021.drop_duplicates()

shares_reported_2021[shares_reported_2021['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
140,USA,0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.

In [12]:
excluded_2021 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2021 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2021 = excluded_2021[excluded_2021['year'] == 2021]
excluded_2021 = excluded_2021[excluded_2021['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IRL', 'KOR', 'MAC', 'MUS', 'MAR' 'NZL', 'POL', 'SWE', 'GBR'])]

# Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2021 = excluded_2021.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2021 with excluded_2021. 
excluded_jurisdictions_2021 = pd.merge(iso_combinations_2021, excluded_2021, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2021 = excluded_jurisdictions_2021.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2021 = excluded_jurisdictions_2021.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2021 = excluded_jurisdictions_2021[excluded_jurisdictions_2021['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IRL', 'KOR', 'MAC', 'MUS', 'MAR' 'NZL', 'POL', 'SWE', 'GBR'])]

excluded_jurisdictions_2021[excluded_jurisdictions_2021['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
830,AUT,2021,USA,"2,069,263.00","726,297,147,483.00","401,888,660,314.00","22,529,811,546.62","270,964,405,217.00","951,640,549,767.00","224,852,493,775.00",580.00,"63,174,139,580.00"
3362,CZE,2021,USA,"444,663.00","238,000,000,000.00","53,690,818,054.00","2,606,664,163.22","30,586,349,491.00","365,000,000,000.00","125,750,238,269.00",0.00,"5,814,794,701.00"
4206,FIN,2021,USA,"985,297.00","817,761,022,659.00","184,844,536,829.00","10,156,173,245.81","612,898,954,550.00","1,130,883,735,340.00","311,960,968,624.00",217.00,"42,314,748,301.00"
4628,GBR,2021,USA,"10,746,171.00","4,880,906,637,939.00","3,615,569,195,881.00","112,780,002,636.05","13,839,749,415,401.00","7,180,000,000,000.00","2,297,536,542,295.00","3,590.00","604,216,148,740.00"
5261,HUN,2021,USA,"134,329.00","58,109,599,064.00","19,550,252,423.00","959,278,134.14","15,860,548,301.00","74,640,793,509.00","16,531,194,445.00",0.00,"6,868,840,612.00"
5894,IRL,2021,USA,"1,915,063.00","427,571,530,900.00","217,000,000,000.00","7,126,386,684.12","2,369,000,000,000.00","697,000,000,000.00","269,410,895,368.00",376.00,"36,519,731,063.00"
6527,KOR,2021,USA,"6,345,530.00","3,019,577,465,309.00","2,541,934,539,062.00","77,256,150,290.88","1,089,902,961,198.00","4,316,949,217,844.00","1,298,930,513,223.00","1,516.00","276,597,253,936.00"
7793,MUS,2021,USA,"101,419.00","170,242,775,061.00","52,766,072,303.00","136,773,695.63","62,329,244,580.00","187,866,781,453.00","17,624,006,392.00",75.00,"7,439,064,276.00"
10325,SWE,2021,USA,"2,924,649.00","868,921,664,525.00","409,722,741,330.00","22,514,030,335.37","336,932,617,882.00","1,252,041,349,614.00","382,298,181,710.00","1,274.00","129,576,740,443.00"


### Step 4.5. Merge with the shares reported, and multiply the number

In [13]:
# Merge with share_reported_2021
excluded_jurisdictions_share_reported_2021 = pd.merge(excluded_jurisdictions_2021, shares_reported_2021, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2021['n_employees'] = excluded_jurisdictions_share_reported_2021['n_employees'] * excluded_jurisdictions_share_reported_2021['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2021['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2021['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2021['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2021['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2021['payroll'] = excluded_jurisdictions_share_reported_2021['payroll'] * excluded_jurisdictions_share_reported_2021['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2021['stated_capital'] = excluded_jurisdictions_share_reported_2021['stated_capital'] * excluded_jurisdictions_share_reported_2021['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2021['total_revenues'] = excluded_jurisdictions_share_reported_2021['total_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['related_party_revenues'] = excluded_jurisdictions_share_reported_2021['related_party_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2021['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2021['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2021['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2021['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2021['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2021 = excluded_jurisdictions_share_reported_2021.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_share_reported_2021[excluded_jurisdictions_share_reported_2021['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
197,AUT,2021,USA,"395,200.37","198,384,558,241.92","88,382,568,513.57","8,279,413,048.09","55,739,216,511.75","243,376,521,454.14","47,859,082,489.78",38.80,"17,020,131,292.33",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
408,CZE,2021,USA,"84,924.43","65,008,550,598.34","11,807,579,744.85","957,915,215.59","6,291,819,603.45","93,346,621,634.09","26,765,507,134.86",0.00,"1,566,599,401.38",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
619,FIN,2021,USA,"188,177.98","223,367,474,028.89","40,650,649,182.00","3,732,261,724.25","126,077,473,165.25","289,216,920,972.42","66,399,822,747.38",14.52,"11,400,275,120.06",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
830,GBR,2021,USA,"2,052,368.76","1,333,196,076,210.98","795,128,909,387.10","41,445,185,791.12","2,846,930,350,558.64","1,836,243,132,418.60","489,022,776,909.91",240.15,"162,785,567,779.30",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1041,HUN,2021,USA,"25,654.97","15,872,356,348.74","4,299,453,293.56","352,522,251.87","3,262,622,391.44","19,088,947,699.05","3,518,608,067.52",0.00,"1,850,576,356.39",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1252,IRL,2021,USA,"365,750.32","116,789,098,743.36","47,722,215,781.01","2,618,854,524.20","487,319,372,485.76","178,253,685,695.79","57,343,185,519.50",25.15,"9,839,004,085.93",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1463,KOR,2021,USA,"1,211,907.71","824,783,001,845.11","559,017,274,536.02","28,390,631,561.21","224,200,433,567.49","1,104,034,589,730.63","276,472,906,913.95",101.41,"74,519,757,742.42",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1674,MUS,2021,USA,"19,369.61","46,500,991,834.26","11,604,211,467.10","50,262,556.25","12,821,548,483.00","48,045,833,879.20","3,751,209,344.20",5.02,"2,004,203,800.98",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1885,SWE,2021,USA,"558,567.16","237,341,756,278.42","90,105,424,295.56","8,273,613,657.98","69,309,325,418.42","320,202,274,336.05","81,370,857,431.80",85.22,"34,910,062,082.90",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [14]:
final_misalignment_2021 = cbcr_sample[cbcr_sample['year'] == 2021].copy()

# Concatenate excluded_jurisdictions_dataset_2021
final_misalignment_2021 = pd.concat([final_misalignment_2021, excluded_jurisdictions_dataset_2021])


final_misalignment_2021[final_misalignment_2021['iso_partner'] == 'USA']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
140,ARE,United Arab Emirates,USA,United States,2021,"22,510,101,353.00","-566,015,448.00",NaN,"33,594,827.00","41,196,000.00","28,384.00","11,019,650,222.00","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00,28.00,28.00,438.00,"-566,015,448.00",0.00,23.84,10.25,23.12,24.25,23.98,21.93,3.00,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","1,566,840,057.22",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
248,ARG,Argentina,USA,United States,2021,"113,406,031.00","67,155,125.00",NaN,"940,629.00","1,346,224.00",114.00,"103,940,740.00","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00,16.00,16.00,38.00,"67,155,125.00",18.02,18.55,4.74,18.46,21.29,18.70,16.77,1.10,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","6,292,973.74",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
720,AUS,Australia,USA,United States,2021,"57,209,834,217.00","8,095,396,499.00",NaN,"379,083,911.00","995,838,351.00","89,401.00","37,039,556,556.00","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00,83.00,83.00,"1,198.00","8,095,396,499.00",22.81,24.77,11.40,24.34,25.88,25.02,23.53,3.33,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","4,935,071,447.12",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
830,AZE,Azerbaijan,USA,United States,2021,"748,376,581.00","-5,934,134.00",NaN,"8,850.00","4,774.00",21.00,"7,712,734.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN,2.00,2.00,3.00,"-5,934,134.00",0.00,20.43,3.09,15.86,16.81,20.44,15.09,NaN,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","1,159,232.00",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
997,BEL,Belgium,USA,United States,2021,"43,812,500,000.00","2,376,700,000.00",NaN,"559,700,000.00","807,700,000.00","58,400.00","14,989,500,000.00","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00,48.00,48.00,340.00,"2,376,700,000.00",21.59,24.50,10.98,23.43,26.66,24.73,23.12,3.26,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","3,223,769,001.60",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
1060,BHR,Bahrain,USA,United States,2021,"31,352,002.00","28,628,903.00",NaN,"1,933,590.00","-7,006,209.00",46.00,"494,407.00","1,253,597.00","491,165,142.00","459,813,140.00",NaN,2.00,2.00,2.00,"28,628,903.00",17.17,17.26,3.85,13.11,14.04,20.01,19.95,NaN,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","2,539,270.10",8.4

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed



In [15]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2021 = final_misalignment_2021[final_misalignment_2021['year'] == 2021].copy()
misalignment_final_estimates_2021 = calculate_misalignment(misalignment_final_estimates_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])


# Perform the groupby operation on 'iso_partner'
country_results_2021 = misalignment_final_estimates_2021.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2021['negative_misalignment'] = -country_results_2021['negative_misalignment'] / 1e6
country_results_2021['positive_misalignment'] = country_results_2021['positive_misalignment'] / 1e6
country_results_2021['theoretical_profit'] = country_results_2021['theoretical_profit'] / 1e6
country_results_2021['reported_profit'] = country_results_2021['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2021 = country_results_2021.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2021['tax_revenue_loss'] = country_results_2021['negative_misalignment'] * country_results_2021['cit']
country_results_2021['tax_revenue_gain'] = country_results_2021['positive_misalignment'] * country_results_2021['etr_average_corrected']

country_results_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2021['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2021['tax_revenue_loss'] / (country_results_2021['gvt_health_expenditure'] / 1e6)
)
    
country_results_2021['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2021['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2021['tax_revenue_loss'] / (country_results_2021['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2021['positive_misalignment'].sum()
total_negative_misalignment = country_results_2021['negative_misalignment'].sum()
total_profits = country_results_2021['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2021['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2021['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2021['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2021}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2021['tax_revenue_loss_caused_pct_of_total'] = country_results_2021['positive_misalignment'] / total_positive_misalignment
country_results_2021['tax_revenue_loss_caused_usd'] = country_results_2021['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2021['tax_revenue_loss_suffered_pct_of_total'] = country_results_2021['tax_revenue_loss'] / total_tax_revenue_loss

country_results_2021 = country_results_2021.sort_values(by='iso_partner')
country_results_2021.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2021.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2021,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2021.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2021: Positive Misalignment: 1420180.0438085557, Negative Misalignment: 1420180.0438085552, Shifted of total profits: 0.15895261084432838, Total tax revenue loss: 347578.9136467748, Total tax revenue gain: 118593.46762131003


/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_12769/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


## Step 6. Checking the datasets

### Step 6.1. Checking the countries

In [16]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2021_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2021.csv')

sotj_2021_countries[sotj_2021_countries['iso_partner'] == 'FRA']

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
63,FRA,"70,272.44",0.00,"264,502.72","180,570.81",France,0.19,0.28,"709,159,262,410.30","275,291,141,343.09",Europe,0.00,1.00,0.00,0.00,"19,962.64",0.00,0.07,0.03,0.00,0.00,0.06


### Step 6.2. Checking the aggregate results

In [17]:
sotj_2021_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2021.csv')
sotj_2021_aggregate

,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2021,"1,420,180.04","1,420,180.04","8,934,612.88",0.16,"347,578.91","118,593.47",0.41,0.06


In [19]:


# Initialize a list to store the aggregate results
results_sample_1 = []

# Start the estimates
misalignment_final_estimates_spain_2021 = final_misalignment_2021[final_misalignment_2021['year'] == 2021].copy()
misalignment_final_estimates_spain_2021 = calculate_misalignment(misalignment_final_estimates_spain_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_spain_2021['negative_misalignment'] = misalignment_final_estimates_spain_2021['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_spain_2021['positive_misalignment'] = misalignment_final_estimates_spain_2021['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_spain_2021 = misalignment_final_estimates_spain_2021.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_spain_2021['negative_misalignment'] = -misalignment_final_estimates_spain_2021['negative_misalignment'] / 1e6
misalignment_final_estimates_spain_2021['positive_misalignment'] = misalignment_final_estimates_spain_2021['positive_misalignment'] / 1e6
misalignment_final_estimates_spain_2021['theoretical_profit'] = misalignment_final_estimates_spain_2021['theoretical_profit'] / 1e6
misalignment_final_estimates_spain_2021['reported_profit'] = misalignment_final_estimates_spain_2021['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_spain_2021['tax_revenue_loss'] = misalignment_final_estimates_spain_2021['negative_misalignment'] * misalignment_final_estimates_spain_2021['cit']
misalignment_final_estimates_spain_2021['tax_revenue_gain'] = misalignment_final_estimates_spain_2021['positive_misalignment'] * misalignment_final_estimates_spain_2021['etr_average_corrected']

#misalignment_final_estimates_spain_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
#    misalignment_final_estimates_spain_2021['gvt_health_expenditure'] == 0, 
#    np.nan, 
#    misalignment_final_estimates_spain_2021['tax_revenue_loss'] / (misalignment_final_estimates_spain_2021['gvt_health_expenditure'] / 1e6)
#)
    
#misalignment_final_estimates_spain_2021['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
#    misalignment_final_estimates_spain_2021['tax_revenue_current_usd'] == 0, 
#    np.nan, 
 #   misalignment_final_estimates_spain_2021['tax_revenue_loss'] / (misalignment_final_estimates_spain_2021['tax_revenue_current_usd'] / 1e6)
#)

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_spain_2021 = misalignment_final_estimates_spain_2021[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Calculate totals
#total_positive_misalignment = misalignment_final_estimates_spain_2021['positive_misalignment'].sum()
#total_negative_misalignment = misalignment_final_estimates_spain_2021['negative_misalignment'].sum()
#total_profits = misalignment_final_estimates_spain_2021['reported_profit'].sum()
#misaligned_of_total_profits = total_positive_misalignment / total_profits
#total_tax_revenue_loss = misalignment_final_estimates_spain_2021['tax_revenue_loss'].sum()
#total_tax_revenue_gain = misalignment_final_estimates_spain_2021['tax_revenue_gain'].sum()
#average_tax_revenue_loss_pct_of_gvt_health_expenditure = misalignment_final_estimates_spain_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
#average_tax_revenue_loss_pct_of_total_tax_revenues = misalignment_final_estimates_spain_2021['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

#print(f"Year {2021}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
#        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

#misalignment_final_estimates_spain_2021 = misalignment_final_estimates_spain_2021.sort_values(by='iso_partner')
#misalignment_final_estimates_spain_2021.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_spain_2021.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_spain_2021 = misalignment_final_estimates_spain_2021.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_spain_2021[(misalignment_final_estimates_spain_2021['iso_parent'] == 'ESP') | (misalignment_final_estimates_spain_2021['iso_partner'] == 'ESP')]

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_12769/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
84,ARE,ESP,2021,0.25000000,89.20950884,466.99609000,-0.00000000,377.78658116,-0.00000000,45.75008356
316,ARG,ESP,2021,0.25000000,4.14703067,25.46038400,-0.00000000,21.31335333,-0.00000000,2.58105434
380,AUS,ESP,2021,0.25000000,122.91634007,52.68373000,21.53690691,0.00000000,5.38422673,0.00000000
115,AUT,ESP,2021,NaN,"1,163.40571134",628.39862830,535.00708304,0.00000000,NaN,NaN
592,BEL,ESP,2021,0.25000000,574.77579017,145.80000000,363.98492475,0.00000000,90.99623119,0.00000000
762,BMU,ESP,2021,0.25000000,342.89179705,157.16593700,162.41320318,0.00000000,40.60330080,0.00000000
926,BRA,ESP,2021,0.25000000,180.78685266,932.64429800,-0.00000000,751.85744534,-0.00000000,91.05019253
972,CAN,ESP,2021,0.25000000,"1,237.04389737",0.00000000,"1,237.04389737",0.00000000,309.26097434,0.00000000
1018,CHE,ESP,2021,0.25000000,"2,452.99612749","2,623.69085300",-0.00000000,170.69472551,-0.00000000,20.67118936
1164,CHN,ESP,2021,0.25000000,"1,530.11408413",197.95274900,"1,185.34545732",0.00000000,296.33636433,0.00000000
